#SILVER

Read Bronze Data

In [0]:
df = spark.read.table("workspace.bronze.airbnb_listings_amsterdam")
display(df.limit(5))

id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,ingestion_date
27886,"Romantic, stylish B&B houseboat in canal district",97647,Flip,null,Centrum-West,52.38761,4.89188,Private room,132,3,311,2025-09-07,1.87,1,17,33,0363 974D 4986 7411 88D8,2026-04-12T13:28:14.433Z
28871,Comfortable double room,124245,Edwin,null,Centrum-West,52.36775,4.89092,Private room,89,2,732,2025-09-07,3.99,2,126,93,0363 607B EA74 0BD8 2F6F,2026-04-12T13:28:14.433Z
29051,Comfortable single / double room,124245,Edwin,null,Centrum-Oost,52.36584,4.89111,Private room,61,2,849,2025-09-08,4.81,2,95,86,0363 607B EA74 0BD8 2F6F,2026-04-12T13:28:14.433Z
44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,Jan,null,Centrum-Oost,52.37168,4.91471,Entire home/apt,null,3,42,2022-08-20,0.23,1,0,0,0363 E76E F06A C1DD 172C,2026-04-12T13:28:14.433Z
48373,Cozy family home in Amsterdam South,220434,Vesna & Misha,null,Buitenveldert - Zuidas,52.327807756778164,4.87680005722526,Entire home/apt,null,3,5,2024-04-28,0.19,1,0,0,0363 4A2B A6AD 0196 F684,2026-04-12T13:28:14.433Z


Data Transformation

In [0]:
from pyspark.sql.functions import col, trim,to_date

df_silver = df.select(
    col("id").cast("long"),
    col("host_id").cast("long"),
    trim(col("host_name")).alias("host_name"),
    trim(col("neighbourhood")).alias("neighbourhood"),
    trim(col("room_type")).alias("room_type"),
    col("price").cast("float"),
    col("minimum_nights").cast("int"),
    col("number_of_reviews").cast("int"),
    col("reviews_per_month").cast("double"),
    col("availability_365").cast("int"),
to_date(col("ingestion_date")).alias("ingestion_date")
)
df_silver = df_silver.filter(
    (col("price").isNotNull()) &
    (col("minimum_nights") > 0)
)
display(df_silver.limit(5))

id,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,availability_365,ingestion_date
27886,97647,Flip,Centrum-West,Private room,132.0,3,311,1.87,17,2026-04-12
28871,124245,Edwin,Centrum-West,Private room,89.0,2,732,3.99,126,2026-04-12
29051,124245,Edwin,Centrum-Oost,Private room,61.0,2,849,4.81,95,2026-04-12
49552,225987,Joanna & MP,Centrum-West,Entire home/apt,322.0,3,609,3.36,223,2026-04-12
50263,230246,Donald,Centrum-Oost,Entire home/apt,457.0,2,177,0.97,354,2026-04-12


Data Quality 

In [0]:
from pyspark.sql.functions import sum, when
print("Nulls by Column:")
display(
    df_silver.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_silver.columns
    ])
)

Nulls by Column:


id,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,availability_365,ingestion_date
0,0,2,0,0,0,0,0,656,0,0


Id Duplicates

In [0]:
from pyspark.sql.functions import col

df_silver.groupBy("id").count().filter(col("count") > 1).show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



Bronze Versus Silver

In [0]:
bronze_count = spark.read.table("workspace.bronze.airbnb_listings_amsterdam").count()
silver_count = df_silver.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Rows excluded: {bronze_count - silver_count}")

Bronze rows: 10480
Silver rows: 5874
Rows excluded: 4606


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.airbnb_listings_amsterdam")
    
display(df_silver.limit(5))

id,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,availability_365,ingestion_date
27886,97647,Flip,Centrum-West,Private room,132.0,3,311,1.87,17,2026-04-12
28871,124245,Edwin,Centrum-West,Private room,89.0,2,732,3.99,126,2026-04-12
29051,124245,Edwin,Centrum-Oost,Private room,61.0,2,849,4.81,95,2026-04-12
49552,225987,Joanna & MP,Centrum-West,Entire home/apt,322.0,3,609,3.36,223,2026-04-12
50263,230246,Donald,Centrum-Oost,Entire home/apt,457.0,2,177,0.97,354,2026-04-12


Validation

In [0]:
df_silver_rows=df_silver.count()
table_silver_rows=spark.table("workspace.silver.airbnb_listings_amsterdam").count()
validation= "Passed" if df_silver_rows==table_silver_rows else "Failed"
print(f"""
Data Insertion:{validation}
Table Rows:{table_silver_rows}
Dataframe Rows:{df_silver_rows}
     """)


Data Insertion:Passed
Table Rows:5874
Dataframe Rows:5874
     
